# Clase 2: Optimizacion y Entrenamiento de Redes Neuronales

## Caso de Estudio: Prediccion de Supervivencia con el Titanic Dataset

**Modulo:** Cientifico de Datos e Inteligencia Artificial Aplicada
**Tecnologias:** Python, TensorFlow, Keras, Scikit-Learn, Matplotlib
**Dataset:** [Titanic - Machine Learning from Disaster (Kaggle)](https://www.kaggle.com/competitions/titanic)

## Objetivo de esta practica

En la Clase 1 construimos una ANN que funcionaba. Ahora vamos a **entrenarla bien**.

Partimos de un modelo baseline y medimos, con evidencia, el impacto de cada decision de entrenamiento:
optimizador -> learning rate -> batch size -> inicializacion -> regularizacion -> callbacks.

Flujo: datos de Kaggle -> baseline -> experimentos comparativos -> modelo final optimizado -> evaluacion.

## 0) Obtener el dataset desde Kaggle

Los datasets **no se versionan en el repositorio**. Cada estudiante los descarga en su entorno.

### Opcion A: descarga manual
1. Crear cuenta en https://www.kaggle.com
2. Entrar a https://www.kaggle.com/competitions/titanic y aceptar las reglas (`Join Competition`).
3. Pestana `Data` -> descargar `train.csv`.
4. Guardarlo en la carpeta `data/` de esta clase: `data/train.csv`.

### Opcion B: API oficial de Kaggle
```bash
pip install kaggle
# Token: https://www.kaggle.com/settings -> API -> Create New Token
# Windows: C:\Users\<usuario>\.kaggle\kaggle.json
# Linux/macOS: ~/.kaggle/kaggle.json   (chmod 600 ~/.kaggle/kaggle.json)
kaggle competitions download -c titanic -p data
```
Luego descomprimir `data/titanic.zip`.

> Recuerda anadir `data/` y `kaggle.json` al `.gitignore`.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

print('TensorFlow:', tf.__version__)

TensorFlow: 2.10.0


In [3]:
# 1) Cargar el dataset Titanic descargado de Kaggle
CANDIDATE_PATHS = [
    os.path.join('titanic', 'train.csv'),
    os.path.join('data', 'train.csv'),
    'train.csv',
    os.path.join('data', 'titanic', 'train.csv'),
]

file_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)

if file_path is None:
    raise FileNotFoundError(
        'No se encontro train.csv.\n'
        'Descargalo desde https://www.kaggle.com/competitions/titanic (pestana Data)\n'
        'y guardalo como titanic/train.csv o data/train.csv dentro de la carpeta de esta clase.'
    )

df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

print('Archivo cargado:', file_path)
print('Shape:', df.shape)
df.head()

Archivo cargado: titanic\train.csv
Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
# 2) Exploracion rapida
print(df.info())
print('\nValores nulos por columna:')
print(df.isna().sum()[df.isna().sum() > 0])
print('\nDistribucion del target (Survived):')
print(df['Survived'].value_counts(normalize=True).round(3))

### Lectura del EDA

- `Age` y `Embarked` tienen nulos: hay que imputar, no eliminar filas (perderiamos ~20% del dataset).
- `Cabin` tiene demasiados nulos para ser util en este baseline.
- `Name`, `Ticket` y `PassengerId` son identificadores: se descartan para evitar ruido y leakage.
- El target esta moderadamente desbalanceado (~62% / 38%), aceptable sin `class_weight`.

In [ ]:
# 3) Preparar X e y
target_col = 'Survived'
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin', target_col]

X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df[target_col].astype(int)

numeric_cols = ['Age', 'SibSp', 'Parch', 'Fare']
categorical_cols = ['Pclass', 'Sex', 'Embarked']

print('Numericas  :', numeric_cols)
print('Categoricas:', categorical_cols)
print('X shape    :', X.shape)

In [ ]:
# 4) Split + preprocesamiento (imputacion + escalado + One-Hot)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:  # scikit-learn < 1.2
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', ohe),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_cols),
    ('cat', categorical_pipe, categorical_cols),
])

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

X_train = np.asarray(X_train, dtype='float32')
X_test = np.asarray(X_test, dtype='float32')
y_train_np = y_train.to_numpy()
y_test_np = y_test.to_numpy()

INPUT_DIM = X_train.shape[1]
print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('Features tras preprocesamiento:', INPUT_DIM)

> **Importante:** el preprocesador se ajusta (`fit`) solo con `X_train`. Si se ajustara con todo el
> dataset, la media y la desviacion de test se filtrarian al entrenamiento (**data leakage**).

## 5) Funciones de apoyo para experimentar

Para comparar decisiones de entrenamiento de forma justa necesitamos que **todo lo demas se mantenga igual**:
misma arquitectura, misma semilla, mismas epocas. Estas dos funciones nos dan esa base controlada.

In [ ]:
def build_model(optimizer='adam',
                hidden=(16, 8),
                activation='relu',
                initializer='glorot_uniform',
                dropout=0.0,
                l2=0.0,
                batchnorm=False,
                seed=SEED):
    """Construye una ANN binaria configurable para los experimentos de la clase."""
    keras.utils.set_random_seed(seed)

    model = keras.Sequential(name='ann_titanic')
    model.add(keras.Input(shape=(INPUT_DIM,)))

    for units in hidden:
        model.add(layers.Dense(
            units,
            activation=None if batchnorm else activation,
            kernel_initializer=initializer,
            kernel_regularizer=regularizers.l2(l2) if l2 > 0 else None,
        ))
        if batchnorm:
            model.add(layers.BatchNormalization())
            model.add(layers.Activation(activation))
        if dropout > 0:
            model.add(layers.Dropout(dropout, seed=seed))

    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(optimizer=optimizer,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model


def run_experiment(label, epochs=80, batch_size=32, model_kwargs=None, fit_kwargs=None):
    """Entrena un modelo y devuelve (label, history, model, val_loss_minima)."""
    model = build_model(**(model_kwargs or {}))
    history = model.fit(
        X_train, y_train_np,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        **(fit_kwargs or {})
    )
    best_val_loss = float(np.min(history.history['val_loss']))
    best_val_acc = float(np.max(history.history['val_accuracy']))
    print(f'{label:<28} val_loss min: {best_val_loss:.4f} | val_acc max: {best_val_acc:.4f}')
    return {'label': label, 'history': history, 'model': model,
            'val_loss': best_val_loss, 'val_acc': best_val_acc}


def plot_histories(results, title, metric='loss'):
    """Compara varias corridas en una sola figura (train vs validacion)."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for res in results:
        h = res['history'].history
        axes[0].plot(h[metric], label=res['label'])
        axes[1].plot(h['val_' + metric], label=res['label'])

    axes[0].set_title(f'{title} - {metric} (train)')
    axes[1].set_title(f'{title} - {metric} (validacion)')
    for ax in axes:
        ax.set_xlabel('Epoca')
        ax.set_ylabel(metric)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


def comparison_table(results):
    return (pd.DataFrame([{'Configuracion': r['label'],
                           'Val loss (min)': round(r['val_loss'], 4),
                           'Val accuracy (max)': round(r['val_acc'], 4)}
                          for r in results])
            .sort_values('Val loss (min)')
            .reset_index(drop=True))

In [ ]:
# 6) Modelo baseline: nuestra referencia para todos los experimentos
baseline = run_experiment('Baseline (Adam, lr=0.001)', epochs=80, batch_size=32)
baseline['model'].summary()

## 7) Descenso de gradiente y optimizadores

El entrenamiento actualiza los pesos siguiendo el gradiente de la perdida:

$$
w \leftarrow w - \eta \frac{\partial L}{\partial w}
$$

**Backpropagation** es el algoritmo que calcula esos gradientes hacia atras, capa por capa.
El **optimizador** decide como se usa el gradiente:

| Optimizador | Idea central |
|---|---|
| `SGD` | Aplica la regla base tal cual |
| `SGD + Momentum` | Acumula la direccion previa: atraviesa zonas planas |
| `RMSprop` | Learning rate adaptativo por parametro |
| `Adam` | Momentum + RMSprop: el default razonable |

In [ ]:
# 7.1) Comparativa de optimizadores (misma arquitectura, misma semilla)
opt_results = [
    run_experiment('SGD (lr=0.01)',
                   model_kwargs={'optimizer': keras.optimizers.SGD(learning_rate=0.01)}),
    run_experiment('SGD + Momentum (0.9)',
                   model_kwargs={'optimizer': keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)}),
    run_experiment('RMSprop (lr=0.001)',
                   model_kwargs={'optimizer': keras.optimizers.RMSprop(learning_rate=0.001)}),
    run_experiment('Adam (lr=0.001)',
                   model_kwargs={'optimizer': keras.optimizers.Adam(learning_rate=0.001)}),
]

plot_histories(opt_results, 'Optimizadores', metric='loss')
comparison_table(opt_results)

### Como leer esta comparativa

- **SGD puro** baja la loss de forma lenta y suave: necesita mas epocas o un learning rate mayor.
- **Momentum** acelera claramente a SGD con el mismo learning rate.
- **RMSprop y Adam** convergen rapido en las primeras epocas, pero suelen empezar a sobreajustar antes.
- La curva que importa para decidir es la de **validacion**, no la de train.

## 8) Learning rate: el hiperparametro mas critico

| Learning rate | Sintoma |
|---|---|
| Muy alto | La loss oscila, sube o se vuelve `NaN` |
| Muy bajo | La loss baja demasiado lento, no llega a converger |
| Adecuado | Baja rapido al inicio y luego se estabiliza |

In [ ]:
# 8.1) Barrido de learning rate con Adam
lr_results = [
    run_experiment(f'Adam lr={lr}',
                   model_kwargs={'optimizer': keras.optimizers.Adam(learning_rate=lr)})
    for lr in [0.5, 0.05, 0.001, 0.00001]
]

plot_histories(lr_results, 'Learning rate', metric='loss')
comparison_table(lr_results)

## 9) Schedulers: learning rate que cambia en el tiempo

La estrategia es dar **pasos grandes al inicio** para avanzar rapido y **pasos pequenos al final**
para afinar sin saltarse el minimo.

- `ExponentialDecay`: decae de forma continua segun los pasos ejecutados.
- `PiecewiseConstantDecay`: escalones definidos manualmente.
- `ReduceLROnPlateau`: reduce el lr **solo** cuando `val_loss` deja de mejorar (callback reactivo).

In [ ]:
# 9.1) Scheduler exponencial vs learning rate fijo
steps_per_epoch = max(1, int(np.ceil(len(X_train) * 0.8 / 32)))

lr_schedule = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.01,
    decay_steps=steps_per_epoch * 10,   # cada 10 epocas
    decay_rate=0.5,                     # se reduce a la mitad
    staircase=True
)

sched_results = [
    run_experiment('Adam lr fijo 0.01',
                   model_kwargs={'optimizer': keras.optimizers.Adam(learning_rate=0.01)}),
    run_experiment('Adam + ExponentialDecay',
                   model_kwargs={'optimizer': keras.optimizers.Adam(learning_rate=lr_schedule)}),
]

plot_histories(sched_results, 'Scheduler', metric='loss')

# Visualizar como decae el learning rate a lo largo del entrenamiento
steps = np.arange(0, steps_per_epoch * 80)
lrs = [float(lr_schedule(s)) for s in steps]

plt.figure(figsize=(6, 3.5))
plt.plot(steps / steps_per_epoch, lrs)
plt.title('Learning rate por epoca (ExponentialDecay)')
plt.xlabel('Epoca')
plt.ylabel('Learning rate')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

comparison_table(sched_results)

## 10) Batch size

Define cuantas muestras se procesan antes de cada actualizacion de pesos.

| Batch size | Efecto |
|---|---|
| Pequeno (8-32) | Mas ruido en el gradiente, a veces mejor generalizacion, mas lento por epoca |
| Mediano (32-128) | Equilibrio habitual |
| Grande (256+) | Rapido y estable, riesgo de converger a minimos que generalizan peor |

In [ ]:
# 10.1) Comparativa de batch size
bs_results = [run_experiment(f'batch_size={bs}', batch_size=bs) for bs in [8, 32, 128, 512]]

plot_histories(bs_results, 'Batch size', metric='loss')
comparison_table(bs_results)

## 11) Inicializacion de pesos

Si todos los pesos inician en el mismo valor, todas las neuronas calculan lo mismo y la red nunca
se diferencia (problema de simetria). Una buena inicializacion mantiene la varianza de la senal
entre capas.

| Inicializador | Recomendado con |
|---|---|
| `glorot_uniform` (Xavier) | `tanh`, `sigmoid` — es el default de Keras |
| `he_normal` | `relu` y variantes |
| `zeros` | **Nunca**: rompe el entrenamiento (se incluye solo para demostrarlo) |

In [ ]:
# 11.1) Efecto del inicializador
init_results = [
    run_experiment('glorot_uniform (default)', model_kwargs={'initializer': 'glorot_uniform'}),
    run_experiment('he_normal (para ReLU)', model_kwargs={'initializer': 'he_normal'}),
    run_experiment('zeros (mal ejemplo)', model_kwargs={'initializer': 'zeros'}),
]

plot_histories(init_results, 'Inicializacion', metric='loss')
comparison_table(init_results)

## 12) Regularizacion: controlar el overfitting

Forzamos primero el overfitting con una red sobredimensionada y luego aplicamos cada tecnica.

### L2 (weight decay)
$$
L_{total} = L + \lambda \sum w^2
$$
Penaliza pesos grandes. L1 (`\lambda \sum |w|`) tiende a llevar pesos a cero.

### Dropout
Desactiva aleatoriamente un porcentaje de neuronas en cada paso de entrenamiento (valores tipicos 0.2-0.5).
En inferencia se desactiva solo.

### Batch Normalization
Normaliza las activaciones por mini-batch: estabiliza el entrenamiento y permite learning rates mas altos.

In [ ]:
# 12.1) Red grande sin regularizar vs regularizada
BIG = (128, 64, 32)

reg_results = [
    run_experiment('Grande sin regularizar',
                   model_kwargs={'hidden': BIG}),
    run_experiment('Grande + L2 (0.01)',
                   model_kwargs={'hidden': BIG, 'l2': 0.01}),
    run_experiment('Grande + Dropout (0.3)',
                   model_kwargs={'hidden': BIG, 'dropout': 0.3}),
    run_experiment('Grande + BatchNorm',
                   model_kwargs={'hidden': BIG, 'batchnorm': True}),
    run_experiment('Grande + L2 + Dropout + BN',
                   model_kwargs={'hidden': BIG, 'l2': 0.001, 'dropout': 0.3, 'batchnorm': True}),
]

plot_histories(reg_results, 'Regularizacion', metric='loss')
comparison_table(reg_results)

### Senal de overfitting

En la red grande sin regularizar la `loss` de train sigue bajando mientras la `val_loss` empieza a
**subir**: el modelo esta memorizando el conjunto de entrenamiento. Las variantes regularizadas
mantienen la brecha entre train y validacion mucho mas estrecha.

## 13) Callbacks: no elegir las epocas a mano

| Callback | Funcion |
|---|---|
| `EarlyStopping` | Detiene el entrenamiento cuando `val_loss` deja de mejorar |
| `ModelCheckpoint` | Guarda el mejor modelo segun la metrica de validacion |
| `ReduceLROnPlateau` | Reduce el learning rate cuando la mejora se estanca |

La practica correcta es entrenar con un limite alto de epocas y dejar que `EarlyStopping`
con `restore_best_weights=True` recupere el mejor punto.

In [ ]:
# 13.1) Modelo final: mejores decisiones + callbacks
cb = [
    callbacks.EarlyStopping(monitor='val_loss', patience=15,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=7, min_lr=1e-5, verbose=1),
    callbacks.ModelCheckpoint('mejor_modelo_titanic.keras', monitor='val_loss',
                              save_best_only=True, verbose=0),
]

final = run_experiment(
    'Modelo final optimizado',
    epochs=300,
    batch_size=32,
    model_kwargs={
        'optimizer': keras.optimizers.Adam(learning_rate=0.005),
        'hidden': (64, 32),
        'initializer': 'he_normal',
        'dropout': 0.3,
        'l2': 0.001,
        'batchnorm': True,
    },
    fit_kwargs={'callbacks': cb},
)

model = final['model']
history = final['history']
print('\nEpocas ejecutadas:', len(history.history['loss']))

In [ ]:
# 14) Evaluacion en test
y_proba = model.predict(X_test, verbose=0).ravel()
y_pred = (y_proba >= 0.5).astype(int)

loss_test, acc_test = model.evaluate(X_test, y_test_np, verbose=0)
print('Test loss    :', round(float(loss_test), 4))
print('Test accuracy:', round(float(acc_test), 4))
print('\nReporte de clasificacion:')
print(classification_report(y_test_np, y_pred, target_names=['No sobrevivio', 'Sobrevivio']))

In [ ]:
# 14.1) Graficas de evaluacion del modelo final
h = history.history
epochs_range = range(1, len(h['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs_range, h['accuracy'], label='Train Accuracy')
axes[0].plot(epochs_range, h['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Accuracy por epoca')
axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('Accuracy')
axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(epochs_range, h['loss'], label='Train Loss')
axes[1].plot(epochs_range, h['val_loss'], label='Val Loss')
axes[1].axvline(int(np.argmin(h['val_loss'])) + 1, color='red', ls='--',
                label='Mejor epoca (min val_loss)')
axes[1].set_title('Loss por epoca')
axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Loss')
axes[1].grid(alpha=0.3); axes[1].legend()

plt.tight_layout()
plt.show()

cm = confusion_matrix(y_test_np, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['No sobrevivio', 'Sobrevivio'])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('Matriz de confusion (Test)')
plt.show()

In [ ]:
# 15) Resumen del experimento: baseline vs modelo final
resumen = comparison_table([baseline, final])
print(resumen.to_string(index=False))

base_loss, base_acc = baseline['model'].evaluate(X_test, y_test_np, verbose=0)
print('\nEn test:')
print(f'  Baseline      -> accuracy {base_acc:.4f} | loss {base_loss:.4f}')
print(f'  Modelo final  -> accuracy {acc_test:.4f} | loss {loss_test:.4f}')

## Como leer estas graficas (diagnostico)

| Situacion | Train | Validacion | Diagnostico | Accion |
|---|---|---|---|---|
| Ambas bajan y quedan cercanas | Loss baja | Loss baja | Buen ajuste | Consolidar |
| Train baja, validacion sube | Loss baja | Loss sube | **Overfitting** | Dropout, L2, EarlyStopping, mas datos |
| Ambas se quedan altas | Loss alta | Loss alta | **Underfitting** | Mas capas/neuronas, mas epocas, mayor lr |
| La loss oscila fuerte | Inestable | Inestable | Learning rate alto | Reducir lr o usar scheduler |
| Validacion mejor que train | - | - | Dropout activo en train | Normal, confirmar en test |

### Matriz de confusion (Test)
- Filas: clase real. Columnas: clase predicha.
- Diagonal principal: aciertos. Fuera de la diagonal: errores.
- En Titanic interesa mirar los **falsos negativos** (pasajeros que sobrevivieron y el modelo
  clasifico como no sobrevivientes): si el costo de ese error es alto, conviene bajar el umbral de 0.5.

### Linea del punto rojo en la curva de Loss
Marca la epoca con `val_loss` minima, que es el modelo que `EarlyStopping` restauro con
`restore_best_weights=True`. Las epocas posteriores solo estaban memorizando.

In [ ]:
# 16) Prediccion de un pasajero de ejemplo
pasajero = pd.DataFrame([{
    'Pclass': 3,
    'Sex': 'male',
    'Age': 22.0,
    'SibSp': 1,
    'Parch': 0,
    'Fare': 7.25,
    'Embarked': 'S',
}])

pasajero_prep = np.asarray(preprocessor.transform(pasajero), dtype='float32')
proba = float(model.predict(pasajero_prep, verbose=0)[0][0])

print('Probabilidad de supervivencia:', round(proba, 4))
print('Clase predicha:', 'Sobrevivio' if proba >= 0.5 else 'No sobrevivio')

## Actividad rapida

1. Cambia el optimizador del modelo final a `SGD(learning_rate=0.01, momentum=0.9)` y compara el
   accuracy en test. Documenta cual gano y por que crees que fue asi.
2. Ajusta el umbral de decision (`0.3`, `0.4`, `0.6`) y analiza como cambian precision y recall
   de la clase `Sobrevivio`.
3. Reduce `patience` de `EarlyStopping` a 3 y explica que riesgo introduce detenerse demasiado pronto.
4. Prueba `dropout=0.6` y justifica con las curvas si ayuda o si provoca underfitting.
5. Crea una variable nueva (por ejemplo `FamilySize = SibSp + Parch + 1`), reentrena el modelo final
   y reporta si el aporte de la variable supera el de los ajustes de optimizacion.

## Conclusion de la clase

La arquitectura define **que puede aprender** la red; la optimizacion define **si realmente lo aprende**.
Un baseline con buenas decisiones de entrenamiento supera a una red grande mal entrenada.